# 00 - Utilidades del Pipeline

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

Este notebook **no se ejecuta solo**: los demás notebooks lo cargan con `%run ./00_Utilidades_Pipeline`. Concentra en un solo lugar:

* Los **nombres de todas las tablas** del pipeline, siguiendo la convención del catálogo `datos_empresas`.
* Las funciones de **calidad de datos**: cálculo del motivo de rechazo y separación aceptados / rechazados.
* La función de **escritura idempotente** (`escribir_merge`), usada por Silver, DQL y Gold.
* El **registro de auditoría** de cada escritura, que sirve como evidencia de idempotencia.

## Convención de nombres

| Capa | Esquema | Patrón | Ejemplo |
|---|---|---|---|
| Bronze (TP1) | `bronze` | `de_semiestructurado_<fuente>_api` | `de_semiestructurado_registromercantil_api` |
| Silver | `silver` | `de_<fuente>` (sin sufijo de origen) | `de_registromercantil` |
| DQL | `dql` | `de_<fuente>_dql` | `de_registromercantil_dql` |
| Gold | `gold` | `de_dim_*`, `de_fact_*`, `de_agg_*` | `de_fact_contratos` |
| Auditoría | `auditoria` | `de_pipeline_*` | `de_pipeline_ejecuciones` |

## Estrategia de idempotencia: MERGE por llave de negocio con detección de cambios

Bronze se carga de forma **incremental (append)** en el TP1: una misma empresa o contrato puede llegar varias veces con versiones nuevas. Por eso todas las tablas se escriben con `MERGE`:

1. Cada fila lleva un `_hash_registro` (SHA-256 de su contenido de negocio).
2. `WHEN MATCHED AND hash distinto -> UPDATE`: solo se reescribe si el contenido cambió.
3. `WHEN NOT MATCHED -> INSERT`: llaves nuevas.
4. `WHEN NOT MATCHED BY SOURCE -> DELETE`: llaves que ya no existen en el resultado (por ejemplo, un registro que antes era válido y ahora se rechaza). Así la tabla siempre refleja exactamente el último cálculo.

Al ejecutar el pipeline dos veces sobre el mismo Bronze, la segunda corrida termina con **0 insertadas, 0 actualizadas y 0 eliminadas**, y `_processed_at` / `_fecha_rechazo` conservan el valor de la primera corrida.

In [ ]:
%python
import json
import time
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "datos_empresas"


def tabla(esquema: str, nombre: str) -> str:
    """Nombre completo catalogo.esquema.tabla."""
    return f"{CATALOGO}.{esquema}.{nombre}"


# --- Bronze: entrada del pipeline (construida en el TP1, no se modifica) ---
BRONZE_CIIU = tabla("bronze", "de_semiestructurado_actividadeseconomicas_api")
BRONZE_RUES = tabla("bronze", "de_semiestructurado_registromercantil_api")
BRONZE_SECOP = tabla("bronze", "de_semiestructurado_secop_contratos_api")
BRONZE_TRM = tabla("bronze", "de_semiestructurado_tasacambio_api")

# --- Silver: datos limpios, tipados y validados ---
SILVER_CIIU = tabla("silver", "de_actividadeseconomicas")
SILVER_RUES = tabla("silver", "de_registromercantil")
SILVER_SECOP = tabla("silver", "de_secop_contratos")
SILVER_TRM = tabla("silver", "de_tasacambio")

# --- DQL: registros rechazados con su motivo ---
DQL_CIIU = tabla("dql", "de_actividadeseconomicas_dql")
DQL_RUES = tabla("dql", "de_registromercantil_dql")
DQL_SECOP = tabla("dql", "de_secop_contratos_dql")
DQL_TRM = tabla("dql", "de_tasacambio_dql")

# --- Gold: modelo analítico ---
GOLD_DIM_EMPRESAS = tabla("gold", "de_dim_empresas")
GOLD_FACT_CONTRATOS = tabla("gold", "de_fact_contratos")
GOLD_AGG_MES = tabla("gold", "de_agg_indicadores_mes")

# --- Auditoría: una fila por (ejecución, tabla) ---
AUDITORIA_EJECUCIONES = tabla("auditoria", "de_pipeline_ejecuciones")

# Relación Bronze -> Silver / DQL, usada para conciliar conteos (Bronze = Silver + DQL)
FUENTES = {
    "CIIU": {"bronze": BRONZE_CIIU, "silver": SILVER_CIIU, "dql": DQL_CIIU, "llaves": ["codigo_ciiu"]},
    "RUES": {"bronze": BRONZE_RUES, "silver": SILVER_RUES, "dql": DQL_RUES, "llaves": ["codigo_camara", "matricula"]},
    "SECOP": {"bronze": BRONZE_SECOP, "silver": SILVER_SECOP, "dql": DQL_SECOP, "llaves": ["id_contrato"]},
    "TRM": {"bronze": BRONZE_TRM, "silver": SILVER_TRM, "dql": DQL_TRM, "llaves": ["fecha_vigencia_desde"]},
}

# Tablas Gold con su llave (grano)
TABLAS_GOLD = {
    GOLD_DIM_EMPRESAS: ["llave_empresa"],
    GOLD_FACT_CONTRATOS: ["id_contrato"],
    GOLD_AGG_MES: ["anio_mes"],
}

# Permite que MERGE agregue columnas nuevas si se modifica un notebook (evolución de esquema)
try:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
except Exception as e:  # algunos entornos serverless no permiten cambiar esta configuración
    print(f"Aviso: no se pudo activar autoMerge ({e.__class__.__name__}); se continúa sin evolución de esquema.")

## Parámetro de ejecución

`id_ejecucion` identifica cada corrida del pipeline en la tabla de auditoría. El Job lo envía como parámetro (`{{job.run_id}}`); al ejecutar un notebook a mano toma el valor `manual`.

In [ ]:
%python
dbutils.widgets.text("id_ejecucion", "manual", "Id de ejecución")
ID_EJECUCION = dbutils.widgets.get("id_ejecucion").strip() or "manual"
print(f"id_ejecucion = {ID_EJECUCION}")

## Funciones de transformación

In [ ]:
%python
FORMATO_FECHA_ISO = "yyyy-MM-dd'T'HH:mm:ss.SSS"


def es_vacio(columna) -> "Column":
    """True si la columna es NULL o texto vacío después de quitar espacios."""
    c = F.col(columna) if isinstance(columna, str) else columna
    return c.isNull() | (F.trim(c.cast("string")) == "")


def texto_limpio(columna: str, mayusculas: bool = False):
    """trim + NULL para vacíos; opcionalmente en mayúsculas (estandariza categorías)."""
    c = F.trim(F.col(columna))
    if mayusculas:
        c = F.upper(c)
    return F.when(c == "", F.lit(None)).otherwise(c)


def a_booleano(columna: str):
    """'Si'/'No' (y variantes) -> BOOLEAN. Cualquier otro valor -> NULL."""
    v = F.upper(F.trim(F.col(columna)))
    return (F.when(v.isin("SI", "SÍ", "S", "TRUE", "1"), F.lit(True))
             .when(v.isin("NO", "N", "FALSE", "0"), F.lit(False)))


def solo_digitos(columna: str):
    """Normaliza un documento de identidad: deja solo dígitos y quita ceros a la izquierda.
    Si queda vacío o era solo ceros ('0000000000000'), devuelve NULL."""
    d = F.regexp_replace(F.col(columna), "[^0-9]", "")
    sin_ceros = F.regexp_replace(d, "^0+", "")
    return F.when(sin_ceros == "", F.lit(None)).otherwise(sin_ceros)


def fecha_iso(columna: str):
    """Texto 'yyyy-MM-ddTHH:mm:ss.SSS' -> DATE. try_to_timestamp devuelve NULL (no falla) con ANSI activado."""
    return F.to_date(F.expr(f"try_to_timestamp(`{columna}`, \"{FORMATO_FECHA_ISO}\")"))


def fecha_yyyymmdd(columna: str):
    """Texto 'yyyyMMdd' -> DATE. Fechas de calendario imposibles (mes 99) quedan NULL."""
    return F.to_date(F.expr(f"try_to_timestamp(`{columna}`, 'yyyyMMdd')"))


def decimal_seguro(columna: str, tipo: str = "DECIMAL(22,2)"):
    """Texto -> número sin fallar con valores mal formados."""
    return F.expr(f"try_cast(`{columna}` AS {tipo})")


def hash_columnas(columnas):
    """SHA-256 del contenido de las columnas. Distingue NULL de texto vacío."""
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit("<null>")) for c in columnas]), 256)

## Funciones de calidad (DQL)

* `preparar_bronze`: le da a cada fila de Bronze un identificador estable (`_id_registro_bronze`) y guarda la fila original completa en `_registro_bronze`. El identificador es el mismo en cada corrida, lo que permite hacer MERGE en DQL sin duplicar.
* `marcar_version`: numera las versiones de cada llave de negocio (1 = la más reciente). Las versiones anteriores incumplen la regla de unicidad.
* `separar_aceptados_rechazados`: evalúa todas las reglas a la vez y construye `_motivo_rechazo` con **todas** las reglas incumplidas, separadas por `; `.

In [ ]:
%python
def preparar_bronze(df: DataFrame) -> DataFrame:
    columnas = df.columns
    df = (df
          .withColumn("_registro_bronze", F.struct(*[F.col(f"`{c}`") for c in columnas]))
          .withColumn("_hash_bronze", hash_columnas(sorted(columnas))))
    # Si existen filas idénticas, el número de ocurrencia las distingue de forma determinista
    w = Window.partitionBy("_hash_bronze").orderBy("_hash_bronze")
    return df.withColumn(
        "_id_registro_bronze",
        F.sha2(F.concat_ws("#", F.col("_hash_bronze"), F.row_number().over(w).cast("string")), 256),
    )


def marcar_version(df: DataFrame, llaves, orden) -> DataFrame:
    """_version = 1 para la versión vigente de cada llave. El desempate por hash hace el resultado determinista."""
    w = Window.partitionBy(*llaves).orderBy(*orden, F.col("_hash_bronze").desc(), F.col("_id_registro_bronze").desc())
    return df.withColumn("_version", F.row_number().over(w))


def separar_aceptados_rechazados(df: DataFrame, reglas):
    """reglas: lista de (nombre_regla, condición_de_rechazo). Devuelve (aceptados, rechazados)."""
    motivo = F.concat_ws("; ", *[F.when(condicion, F.lit(nombre)) for nombre, condicion in reglas])
    df = (df
          .withColumn("_motivo_rechazo", F.when(motivo == "", F.lit(None)).otherwise(motivo))
          .withColumn("_num_reglas_incumplidas",
                      F.when(F.col("_motivo_rechazo").isNull(), F.lit(0))
                       .otherwise(F.size(F.split("_motivo_rechazo", "; ")))))
    return df.filter(F.col("_motivo_rechazo").isNull()), df.filter(F.col("_motivo_rechazo").isNotNull())


def construir_dql(df_rechazados: DataFrame) -> DataFrame:
    """Registro rechazado = fila ORIGINAL de Bronze (sin transformar, para poder corregirla) + motivo + trazabilidad."""
    return (df_rechazados
            .select("_id_registro_bronze", "_motivo_rechazo", "_num_reglas_incumplidas", "_registro_bronze.*")
            .withColumn("_hash_registro", F.sha2(F.concat_ws("||", "_id_registro_bronze", "_motivo_rechazo"), 256)))


def construir_silver(df_aceptados: DataFrame, columnas) -> DataFrame:
    """Selecciona las columnas Silver + linaje hacia Bronze + hash de contenido."""
    return (df_aceptados
            .select(*columnas, "_id_registro_bronze", "_bronze_ingested_at")
            .withColumn("_hash_registro", hash_columnas(list(columnas) + ["_id_registro_bronze"])))

## Escritura idempotente (MERGE) y auditoría

In [ ]:
%python
def _version_actual(nombre_tabla: str):
    if not spark.catalog.tableExists(nombre_tabla):
        return None
    return DeltaTable.forName(spark, nombre_tabla).history(1).select("version").collect()[0]["version"]


def _con_reintentos(funcion, intentos: int = 5):
    """Reintenta ante conflictos de concurrencia de Delta (tareas paralelas escribiendo la auditoría)."""
    for i in range(intentos):
        try:
            return funcion()
        except Exception as e:
            es_conflicto = "CONCURRENT" in (e.__class__.__name__ + str(e)).upper()
            if not es_conflicto or i == intentos - 1:
                raise
            time.sleep(2 * (i + 1))


def registrar_auditoria(nombre_tabla: str, capa: str, valores: dict):
    """Upsert de una fila (id_ejecucion, tabla) en la tabla de auditoría.
    Reintentar la misma tarea con el mismo id_ejecucion reemplaza la fila; no la duplica."""
    fila = {"id_ejecucion": ID_EJECUCION, "capa": capa, "tabla": nombre_tabla, **valores}
    columnas = list(fila.keys())
    df = spark.createDataFrame([tuple(fila[c] for c in columnas)],
                               spark.table(AUDITORIA_EJECUCIONES).select(*columnas).schema)
    actualizar = {c: f"s.{c}" for c in columnas if c not in ("id_ejecucion", "tabla")}

    def _merge():
        (DeltaTable.forName(spark, AUDITORIA_EJECUCIONES).alias("t")
         .merge(df.alias("s"), "t.id_ejecucion = s.id_ejecucion AND t.tabla = s.tabla")
         .whenMatchedUpdate(set=actualizar)
         .whenNotMatchedInsert(values={c: f"s.{c}" for c in columnas})
         .execute())

    _con_reintentos(_merge)


def escribir_merge(df: DataFrame, nombre_tabla: str, llaves, capa: str,
                   columna_fecha: str = "_processed_at", comentario: str = None) -> dict:
    """Escritura idempotente: MERGE por llave de negocio, actualizando solo si cambió _hash_registro.
    Devuelve y registra en auditoría las filas insertadas / actualizadas / eliminadas."""
    # 1. La llave debe ser única en el origen; si no, MERGE fallaría o duplicaría
    duplicadas = df.groupBy(*llaves).count().filter("count > 1").limit(5).collect()
    if duplicadas:
        raise ValueError(f"Llave {llaves} duplicada en el origen de {nombre_tabla}: {duplicadas}")

    df = df.withColumn(columna_fecha, F.current_timestamp())

    # 2. Primera ejecución: se crea la tabla vacía con el esquema del origen (luego se llena con el MERGE)
    if not spark.catalog.tableExists(nombre_tabla):
        df.limit(0).write.format("delta").saveAsTable(nombre_tabla)
        if comentario:
            spark.sql(f"COMMENT ON TABLE {nombre_tabla} IS '{comentario.replace(chr(39), chr(92) + chr(39))}'")

    version_antes = _version_actual(nombre_tabla)
    condicion = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in llaves)

    # 3. MERGE con detección de cambios
    (DeltaTable.forName(spark, nombre_tabla).alias("t")
     .merge(df.alias("s"), condicion)
     .whenMatchedUpdateAll(condition="t._hash_registro <> s._hash_registro")
     .whenNotMatchedInsertAll()
     .whenNotMatchedBySourceDelete()
     .execute())

    # 4. Métricas reales de la operación, tomadas del historial Delta
    ultima = DeltaTable.forName(spark, nombre_tabla).history(1).collect()[0]
    if ultima["version"] == version_antes:  # MERGE sin cambios que no generó versión nueva
        m, operacion = {}, "MERGE (sin commit)"
    else:
        m, operacion = (ultima["operationMetrics"] or {}), ultima["operation"]

    metricas = {
        "version_delta": int(ultima["version"]),
        "operacion": operacion,
        "filas_origen": int(m.get("numSourceRows", 0)),
        "filas_insertadas": int(m.get("numTargetRowsInserted", 0)),
        "filas_actualizadas": int(m.get("numTargetRowsUpdated", 0)),
        "filas_eliminadas": int(m.get("numTargetRowsDeleted", 0)),
        "fecha_escritura": datetime.now(),
    }
    registrar_auditoria(nombre_tabla, capa, metricas)
    print(f"{nombre_tabla}: v{metricas['version_delta']} | insertadas={metricas['filas_insertadas']:,} "
          f"actualizadas={metricas['filas_actualizadas']:,} eliminadas={metricas['filas_eliminadas']:,}")
    return metricas


def conciliar_fuente(nombre_fuente: str) -> dict:
    """Verifica que ninguna fila de Bronze se pierda ni se duplique: Bronze = Silver + DQL."""
    f = FUENTES[nombre_fuente]
    n_bronze = spark.table(f["bronze"]).count()
    n_silver = spark.table(f["silver"]).count()
    n_dql = spark.table(f["dql"]).count()
    print(f"[{nombre_fuente}] Bronze: {n_bronze:,} | Silver: {n_silver:,} | DQL: {n_dql:,}")
    assert n_bronze == n_silver + n_dql, f"[{nombre_fuente}] Bronze ({n_bronze}) != Silver ({n_silver}) + DQL ({n_dql})"
    return {"bronze": n_bronze, "silver": n_silver, "dql": n_dql}


def finalizar(resultado: dict):
    """Devuelve un resumen al orquestador (dbutils.notebook.run) y al Job."""
    dbutils.notebook.exit(json.dumps(resultado, default=str))

In [ ]:
%python
print("Utilidades del pipeline cargadas.")